In [1]:
import numpy as np
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
X_train = load_npz("X_train.npz")
X_test = load_npz("X_test.npz")
y_train = np.load("Y_train.npy")
y_test = np.load("Y_test.npy")

In [3]:
def rows_to_set(X):
    return set(
        tuple(row.indices.tolist() + row.data.tolist())
        for row in X
    )

In [4]:

train_set = rows_to_set(X_train)
test_set = rows_to_set(X_test)
overlap = train_set.intersection(test_set)
len(overlap)

0

In [5]:
batch_size = 500
max_sim_scores = []

In [7]:
for i in range(0, X_test.shape[0], batch_size):
    batch = X_test[i:i+batch_size]
    sims = cosine_similarity(batch, X_train)
    max_sim_scores.extend(sims.max(axis=1).tolist())

In [9]:
max_sim_scores = np.array(max_sim_scores)
print(f"Mean max cosine similarity (test->train): {max_sim_scores.mean():.4f}")
print(f"Samples with similarity > 0.95: {(max_sim_scores > 0.95).sum()}")
print(f"Samples with similarity = 1.00: {(max_sim_scores == 1.0).sum()}")

Mean max cosine similarity (test->train): 0.9999
Samples with similarity > 0.95: 74555
Samples with similarity = 1.00: 1379


## Deep learning training data

In [11]:
from sklearn.model_selection import StratifiedKFold
import pandas as pd

In [12]:
df = pd.read_csv("bert_dataset.csv", index_col=False)

In [13]:
X = df["text"].values
y = df["generated"].values

In [15]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [16]:
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train = set(X[train_idx])
    X_test = set(X[test_idx])
    
    overlap = X_train.intersection(X_test)
    print(f"Fold {fold+1} — exact duplicates: {len(overlap)}")

Fold 1 — exact duplicates: 1
Fold 2 — exact duplicates: 0
Fold 3 — exact duplicates: 0
Fold 4 — exact duplicates: 1
Fold 5 — exact duplicates: 0


In [17]:
# Total duplicates
df["text"].duplicated().sum()

np.int64(1)